# Import Train Data, Drop Columns w Missing & Create Train/Valid datasets

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

housing_training = pd.read_csv("train.csv", index_col="Id")

X = housing_training.drop("SalePrice", axis=1)
y = housing_training["SalePrice"]

columns_with_too_many_missings = [col for col in X.columns if X[col].isnull().sum()/X.shape[0] > 0.005]
X.drop(columns_with_too_many_missings, axis=1, inplace=True)

X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size = 0.2, random_state = 0)

# Create Baseline Score Function to evaluate data treatments

In [2]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

def baseline_model(X_train, y_train, X_valid, y_valid):
    baseline_model = RandomForestRegressor(n_estimators=100, random_state=0)
    baseline_model.fit(X_train, y_train)
    predictions = baseline_model.predict(X_valid)
    return mean_absolute_error(y_valid, predictions)

# Drop categorical columns to get a base line score

In [3]:
print(baseline_model(X_train.select_dtypes(exclude=['object']), y_train, X_valid.select_dtypes(exclude=['object']), y_valid))

17837.82570776256


# Create Numerical Simple Imputer to median

In [4]:
from sklearn.impute import SimpleImputer

numerical_imputer = SimpleImputer(strategy='median')
numerical_columns = [col for col in X_train.columns if X_train[col].dtype != 'object']
#X_train_numerical_cols_w_missings = [col for col in numerical_columns if X_train[col].isnull().any()]
print(numerical_columns) 
#X_train[X_train_numerical_cols_w_missings] = numerical_imputer.fit_transform(X_train[X_train_numerical_cols_w_missings])


['MSSubClass', 'LotArea', 'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF', 'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath', 'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'TotRmsAbvGrd', 'Fireplaces', 'GarageCars', 'GarageArea', 'WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch', 'PoolArea', 'MiscVal', 'MoSold', 'YrSold']


# Create Categorical Pipeline - Change to not exclude any categorical column and see how that goes (low hopes given the concentration of records in a single value of these columns)

In [5]:
# Analyze categorical columns and underlying unique values
categorical_columns = [col for col in X_train.columns if X_train[col].dtype == 'object']

#Identify categorical columns for which X_valid values are a subset of X_train, and only keep them
not_subset_categorical_column = [col for col in categorical_columns if not(X_valid[col].isin(X_train[col]).all())]

#Drop these columns from the exercise for now
X_train.drop(not_subset_categorical_column, axis=1, inplace=True)
X_valid.drop(not_subset_categorical_column, axis=1, inplace=True)

#categorical_columns = list(set(categorical_columns) - set(not_subset_categorical_column))

X_train.shape

(1168, 61)

In [6]:
# Define what columns should be looked at from ordinal and one hot encoding standpoint
high_cardinality_columns = [col for col in categorical_columns if X_train[col].nunique() > 10]
low_cardinality_columns = [col for col in categorical_columns if X_train[col].nunique() <= 10]

print(len(high_cardinality_columns), len(low_cardinality_columns))

3 25


In [7]:
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import OneHotEncoder
import numpy as np

# Create ordinal encoding for high cardinality columns
categorical_ordinal_encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=np.nan)

#Create one hot encoding for low cardinality columns
categorical_one_hot_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

In [8]:
#Transform X_train and X_valid datasets using above imputers


#Categorical Ordinal Imputation
#X_train[high_cardinality_columns] = categorical_ordinal_encoder.fit_transform(X_train[high_cardinality_columns])
#X_valid[high_cardinality_columns] = categorical_ordinal_encoder.transform(X_valid[high_cardinality_columns])   


#Categorical One Hot Encoding Imputation
def one_hot_encoder(X, columns_to_impute, fit = False):
    
    print("X Shape Before One Hot: ", X.shape)    
    print("Number of impute columns: ", len(columns_to_impute))
    print("Sum of X colums to impute nuniques : ", X[columns_to_impute].nunique().sum())
    
    if fit:
        categorical_one_hot_encoder.fit(X[columns_to_impute])

    X_one_hot_encoder_output = categorical_one_hot_encoder.transform(X[columns_to_impute])
    X_one_hot_encoder_output = pd.DataFrame(X_one_hot_encoder_output, columns=categorical_one_hot_encoder.get_feature_names_out())
    X_one_hot_encoder_output.index = X.index
    
    X.drop(columns_to_impute, axis=1, inplace=True)
    X = pd.concat([X, X_one_hot_encoder_output], axis = 1)   
    
    print("X  Shape After One Hot: ", X.shape)

    return X


#X_train = one_hot_encoder(X_train, low_cardinality_columns, fit = True)
#X_valid = one_hot_encoder(X_valid, low_cardinality_columns)

print(X_train.shape)
print(X_valid.shape)

(1168, 61)
(292, 61)


In [9]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

categorical_ordinal_pipeline = Pipeline(steps=[
    ('ordinalimpute', SimpleImputer(strategy="most_frequent")),
    ('ordinal', categorical_ordinal_encoder) ])

categorical_one_hot_pipeline = Pipeline(steps=[
    ('onehotimpute', SimpleImputer(strategy="most_frequent")),
    ('onehot', categorical_one_hot_encoder)])

categorical_transformer = ColumnTransformer(transformers=[
    ('ordinal', categorical_ordinal_pipeline, high_cardinality_columns),
    ('onehot', categorical_one_hot_pipeline, low_cardinality_columns),
    ('numerical', numerical_imputer, numerical_columns)])

print("Pre X Train Shape: ", X_train.shape)
print("Pre X Valid Shape: ", X_valid.shape)

X_train = pd.DataFrame(categorical_transformer.fit_transform(X_train), columns = categorical_transformer.get_feature_names_out())
X_valid = pd.DataFrame(categorical_transformer.transform(X_valid), columns=categorical_transformer.get_feature_names_out())

print("Pos X Train Shape: ", X_train.shape)
print("Pos X Valid Shape: ", X_valid.shape)


Pre X Train Shape:  (1168, 61)
Pre X Valid Shape:  (292, 61)
Pos X Train Shape:  (1168, 163)
Pos X Valid Shape:  (292, 163)


# Get baseline model score after imputation

In [10]:
print(baseline_model(X_train, y_train, X_valid, y_valid))


17184.145924657532


# Investigate columns that were dropped and that were present in X train for pottential benefits

In [11]:
original_train_dataset = housing_training.drop("SalePrice", axis=1)
dropped_columns = []
for col_original in original_train_dataset.columns:
    dropped_column = True
    for col_X_train in X_train.columns:
        if col_original in col_X_train:
            dropped_column = False
            break
    if dropped_column:
        dropped_columns.append(col_original)
            

print("Columns dropped: ", dropped_columns, "\n Number of columns dropped :", len(dropped_columns)) 

X_unexplored_features = housing_training[dropped_columns]
columns_w_missings = []
columns_not_subset = []
other_columns = []
for col in X_unexplored_features.columns:
    if X_unexplored_features[col].isnull().sum()/X_unexplored_features.shape[0] > 0.005:
        #print(col, " - column dropped due too many missing values: ", X_unexplored_features[col].isnull().sum()/X_unexplored_features.shape[0])
        columns_w_missings.append(col)
    elif X_unexplored_features[col].dtype == 'object' :
        #print(col, " - categorical column dropped due to train values not being subset of valid ones", X_unexplored_features[col].value_counts(dropna=False))
        columns_not_subset.append(col)
    else:
        other_columns.append(col)

print("Columns with too many missing values: ", columns_w_missings, "Number :", len(columns_w_missings)) 
print("Columns not subset: ", columns_not_subset, "Number: ", len(columns_not_subset))  
print("Other columns: ", other_columns, "Number: ", len(other_columns))



#print([X_unexplored_features[col].value_counts(dropna=False) for col in X_unexplored_features.select_dtypes(include=['object'])])

Columns dropped:  ['LotFrontage', 'Alley', 'MasVnrType', 'MasVnrArea', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'FireplaceQu', 'GarageType', 'GarageYrBlt', 'GarageFinish', 'GarageQual', 'GarageCond', 'PoolQC', 'Fence', 'MiscFeature'] 
 Number of columns dropped : 18
Columns with too many missing values:  ['LotFrontage', 'Alley', 'MasVnrType', 'MasVnrArea', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'FireplaceQu', 'GarageType', 'GarageYrBlt', 'GarageFinish', 'GarageQual', 'GarageCond', 'PoolQC', 'Fence', 'MiscFeature'] Number : 18
Columns not subset:  [] Number:  0
Other columns:  [] Number:  0


# Study columns not subset to identify potential to include any of them in the data treatment pipeline

In [12]:
print([X_unexplored_features[col].value_counts(dropna=False) for col in X_unexplored_features[columns_not_subset].columns])

[]


# Export results to submit on Kaggle

In [14]:
#Apply data treatments to the whole X dataset
#X Drop columns with too many missings already dropped initially
X_full = X
y_full = y
X_full = pd.DataFrame(categorical_transformer.fit_transform(X_full), columns = categorical_transformer.get_feature_names_out())
model = RandomForestRegressor(n_estimators=100, random_state=0)
model.fit(X_full, y_full)

#Apply data transformations to X_test
housing_testing = pd.read_csv("test.csv", index_col="Id")
X_test = pd.DataFrame(categorical_transformer.transform(housing_testing), columns = categorical_transformer.get_feature_names_out())
X_test.index = housing_testing.index
print(X_test)
predicts = model.predict(X_test)
print(predicts)



output = pd.DataFrame({'Id': X_test.index,
                       'SalePrice': predicts})
output.to_csv('kaggle_housing_submission_1_pipeline_all_categoricals.csv', index=False)


      ordinal__Neighborhood  ordinal__Exterior1st  ordinal__Exterior2nd  \
Id                                                                        
1461                   12.0                  12.0                  13.0   
1462                   12.0                  13.0                  14.0   
1463                    8.0                  12.0                  13.0   
1464                    8.0                  12.0                  13.0   
1465                   22.0                   6.0                   6.0   
...                     ...                   ...                   ...   
2915                   10.0                   5.0                   5.0   
2916                   10.0                   5.0                   5.0   
2917                   11.0                  12.0                  13.0   
2918                   11.0                   6.0                  15.0   
2919                   11.0                   6.0                   6.0   

      onehot__MSZoning_C